# NYC Trip Records

**Libraries**

In [159]:
import pandas as pd
import glob

**Load And Merged**

In [160]:
# 1. Load the Parquet files for January to March 202
file_paths = glob.glob('../data/tlc_trips/yellow_tripdata_2025-0*.parquet')

# Combine the 3 months into one DataFrame (using fastparquet as we discovered earlier)
df_tlc = pd.concat([pd.read_parquet(file, engine='fastparquet') for file in file_paths], ignore_index=True)

**Inspecting Merged Dataset**


In [161]:
# Row Count
print(f"Total raw rows loaded (Jan-Mar): {len(df_tlc):,}\n")
# Column Count
print(f"Total columns in dataset: {len(df_tlc.columns)}\n")

# Column Names and Data Types
print("--- Column Names and Data Types ---")
print(df_tlc.dtypes)
print("\n")

Total raw rows loaded (Jan-Mar): 11,198,026

Total columns in dataset: 20

--- Column Names and Data Types ---
VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object




**Inspecting whether the data is within the coorected month range**

In [162]:
# 1. Verify the exact Start and End dates
min_date = df_tlc['tpep_pickup_datetime'].min()
max_date = df_tlc['tpep_pickup_datetime'].max()

print(f"Earliest trip in dataset: {min_date}")
print(f"Latest trip in dataset:   {max_date}")
print("-" * 50)

Earliest trip in dataset: 2007-12-05 18:45:00
Latest trip in dataset:   2025-04-01 00:00:17
--------------------------------------------------


We inspect inside of the dataset for any invalid trips that are not within from January to March. The output showed data errors where the minimum date for the trips starts at "2007-12-05 18:45:00" and maximum trip start at "2025-04-01 00:00:17" so the dataset needs filtering out first to get the corrected data range for our model building. 

*Getting the Corrected Range*

In [163]:
start_date = '2025-01-01'
end_date = '2025-03-31 23:59:59'

# Keep only the rows that fall exactly within our 3-month window
df_clean = df_tlc[(df_tlc['tpep_pickup_datetime'] >= start_date) & 
                    (df_tlc['tpep_pickup_datetime'] <= end_date)]

min_date = df_clean['tpep_pickup_datetime'].min()
max_date = df_clean['tpep_pickup_datetime'].max()

print(f"Earliest trip in dataset: {min_date}")
print(f"Latest trip in dataset:   {max_date}")
print("-" * 50)

print(f"Row count after date filtering: {len(df_clean):,}")

Earliest trip in dataset: 2025-01-01 00:00:00
Latest trip in dataset:   2025-03-31 23:59:59
--------------------------------------------------
Row count after date filtering: 11,198,001


**Checking Invalid Zone ID's**

In [164]:
# Let's see what the lowest and highest Zone IDs are
min_zone = df_clean['PULocationID'].min()
max_zone = df_clean['PULocationID'].max()

print(f"Lowest Pickup Zone ID: {min_zone}")
print(f"Highest Pickup Zone ID: {max_zone}")

# Find how many trips occurred outside the official 1-263 range
invalid_zones = df_clean[(df_clean['PULocationID'] < 1) | (df_clean['PULocationID'] > 263)]
print(f"Total trips with invalid zones (>263 or <1): {len(invalid_zones):,}")

Lowest Pickup Zone ID: 1
Highest Pickup Zone ID: 265
Total trips with invalid zones (>263 or <1): 27,391


There are some invalid zone ID's present which is not in our range of the study so we remove them up.

**Removing Rows with Invalid Zone ids**

In [165]:
df_clean = df_clean[(df_clean['PULocationID'] >= 1) & (df_clean['PULocationID'] <= 263)]

print(f"Row count after dropping unknown zones: {len(df_clean):,}")

Row count after dropping unknown zones: 11,170,610


So the dataset now has valid pickup dates and valid zone IDs.

**Checking Outliers**

In [166]:
# 1. Count the obvious errors (Negative/Zero)
zero_or_negative_fares = len(df_clean[df_clean['fare_amount'] <= 0])
zero_distance = len(df_clean[df_clean['trip_distance'] <= 0])

print(f"Trips with $0 or negative fares: {zero_or_negative_fares:,}")
print(f"Trips with 0.0 mile distance: {zero_distance:,}")

# 2. Look at the extremes to decide our upper cutoff
print("\n--- Summary Statistics (Looking for extreme maximums) ---")
# We set pandas to format numbers cleanly without scientific notation (e.g., no 1.5e6)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
print(df_clean[['fare_amount', 'trip_distance']].describe())

Trips with $0 or negative fares: 539,327
Trips with 0.0 mile distance: 290,759

--- Summary Statistics (Looking for extreme maximums) ---
       fare_amount  trip_distance
count  11170610.00    11170610.00
mean         17.22           6.18
std         262.30         582.45
min       -1807.60           0.00
25%           8.60           1.00
50%          12.80           1.72
75%          20.50           3.24
max      863372.12      320136.29


In [167]:
print("Fixing the Outliers Problem...")

# 1. Filter out <= 0 fares and distances
df_clean = df_clean[(df_clean['fare_amount'] > 0) & (df_clean['trip_distance'] > 0)]

# 2. Filter out extreme maximums (Capping fare at $500 and distance at 100 miles)
df_clean = df_clean[(df_clean['fare_amount'] <= 500) & (df_clean['trip_distance'] <= 100)]

print(f"Row count after dropping outliers: {len(df_clean):,}")

Fixing the Outliers Problem...
Row count after dropping outliers: 10,389,410


**Fixing the pickup count for resolving ghost riders**

In [168]:
# Validate the Passenger Counts (Ghost Trips vs. Real Trips)
total_raw = len(df_clean)
null_passengers = df_clean['passenger_count'].isnull().sum()
zero_passengers = (df_clean['passenger_count'] == 0).sum()
valid_passengers = (df_clean['passenger_count'] > 0).sum()

# Print the validation results
print("-" * 50)
print(f"Ghost Trips (NULL passenger_count): {null_passengers:,}")
print(f"Ghost Trips (ZERO passenger_count): {zero_passengers:,}")
print("-" * 50)
print(f"Valid Journeys (> 0 passengers): {valid_passengers:,}")

# Calculate the percentage of valid data
percent_valid = (valid_passengers / total_raw) * 100
print(f"\nPercentage of Valid Data: {percent_valid:.2f}%")

--------------------------------------------------
Ghost Trips (NULL passenger_count): 1,767,347
Ghost Trips (ZERO passenger_count): 65,595
--------------------------------------------------
Valid Journeys (> 0 passengers): 8,556,468

Percentage of Valid Data: 82.36%


In [169]:
print(f"Row count BEFORE removing ghost trips: {len(df_clean):,}")

# 1. Drop the rows with NULL passenger counts
df_clean = df_clean.dropna(subset=['passenger_count'])

# 2. Drop the rows with ZERO passenger counts (Keep only > 0)
df_clean = df_clean[df_clean['passenger_count'] > 0]

print(f"Row count AFTER removing ghost trips: {len(df_clean):,}")

Row count BEFORE removing ghost trips: 10,389,410
Row count AFTER removing ghost trips: 8,556,468


**Missing Values**

In [170]:
# Null Value Count for Each Column
print("--- Null Value Count per Column ---")
print(df_clean.isnull().sum())

--- Null Value Count per Column ---
VendorID                 0
tpep_pickup_datetime     0
tpep_dropoff_datetime    0
passenger_count          0
trip_distance            0
RatecodeID               0
store_and_fwd_flag       0
PULocationID             0
DOLocationID             0
payment_type             0
fare_amount              0
extra                    0
mta_tax                  0
tip_amount               0
tolls_amount             0
improvement_surcharge    0
total_amount             0
congestion_surcharge     0
Airport_fee              0
cbd_congestion_fee       0
dtype: int64


**Column Selection that are truely necessary**

In [171]:
# 1. Select ONLY the columns needed for feature extraction and aggregation
final_cols = ['tpep_pickup_datetime', 'PULocationID']

# 2. IMPORTANT: Pull from df_clean (your cleaned data), NOT df_tlc (raw data)
# This keeps your 8,556,468 valid rows and removes the 2007 dates!
df_final = df_clean[final_cols].copy()

# 3. Validation Check for your Lecturer
print(f"Final valid rows: {len(df_final):,}")
print(f"Date Range: {df_final['tpep_pickup_datetime'].min()} to {df_final['tpep_pickup_datetime'].max()}")
print(f"Total Zones: {df_final['PULocationID'].nunique()}")
# Column Names and Data Types
print("--- Column Names and Data Types ---")
print(df_final.dtypes)

Final valid rows: 8,556,468
Date Range: 2025-01-01 00:00:00 to 2025-03-31 23:59:59
Total Zones: 257
--- Column Names and Data Types ---
tpep_pickup_datetime    datetime64[us]
PULocationID                     int32
dtype: object


# Featuring Engineering For Temporal Features

**Temporal Feature Extraction**

In [172]:
# 1. Extract the 'date' for grouping
df_final['date'] = df_final['tpep_pickup_datetime'].dt.date

# 2. Extract the Hour of Day (0 to 23) 
df_final['hour'] = df_final['tpep_pickup_datetime'].dt.hour

# 3. Extract the Day of Week (0=Monday to 6=Sunday)
df_final['day_of_week'] = df_final['tpep_pickup_datetime'].dt.dayofweek

# 4. Extract the Month (1 to 3 for your Jan-Mar window)
df_final['month'] = df_final['tpep_pickup_datetime'].dt.month

# Validation: Check the new features
print(df_final[['tpep_pickup_datetime', 'hour', 'day_of_week', 'month']].head())

  tpep_pickup_datetime  hour  day_of_week  month
0  2025-01-01 00:18:38     0            2      1
1  2025-01-01 00:32:40     0            2      1
2  2025-01-01 00:44:04     0            2      1
3  2025-01-01 00:14:27     0            2      1
4  2025-01-01 00:21:34     0            2      1


**Aggregation to Zone-Hour Level**

In [173]:
# Group by the features and PULocationID, then count the occurrences
# This 'size()' count becomes your TARGET VARIABLE: pickup_count
df_hourly = df_final.groupby(['PULocationID', 'date', 'hour', 'day_of_week', 'month']).size().reset_index(name='pickup_count')

# Final Verification for your report
print(f"Aggregated dataset shape: {df_hourly.shape}")
print(f"Average demand per zone-hour: {df_hourly['pickup_count'].mean():.2f}")
print(df_hourly.head())

Aggregated dataset shape: (214146, 6)
Average demand per zone-hour: 39.96
   PULocationID        date  hour  day_of_week  month  pickup_count
0             1  2025-01-01     6            2      1             1
1             1  2025-01-01    13            2      1             1
2             1  2025-01-01    14            2      1             1
3             1  2025-01-01    16            2      1             1
4             1  2025-01-01    18            2      1             1


# Integrety

Since we got 257 Zones only we havent put in zero values for the missing 263 zones. So in here we will be putiing it in. 

**Create the Master Spatiotemporal Grid**

In [174]:
import pandas as pd

# 1. Define your study range: Jan 1 to March 31, 2025
full_range = pd.date_range(start='2025-01-01', end='2025-03-31 23:00:00', freq='H')

# 2. Define all 263 official Zone IDs
all_zones = range(1, 264)

# 3. Create the "Cartesian Product" (Every zone for every hour)
index = pd.MultiIndex.from_product([all_zones, full_range], names=['PULocationID', 'tpep_pickup_datetime'])
master_grid = pd.DataFrame(index=index).reset_index()

# 4. Extract temporal features for the master grid (to match your df_hourly)
master_grid['date'] = master_grid['tpep_pickup_datetime'].dt.date
master_grid['hour'] = master_grid['tpep_pickup_datetime'].dt.hour
master_grid['day_of_week'] = master_grid['tpep_pickup_datetime'].dt.dayofweek
master_grid['month'] = master_grid['tpep_pickup_datetime'].dt.month

C:\Users\ajakd\AppData\Local\Temp\ipykernel_11048\2499976538.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_range = pd.date_range(start='2025-01-01', end='2025-03-31 23:00:00', freq='H')


Left Join with Zero-Padding

In [175]:
# 1. Merge your current counts onto the master grid
# We merge on PULocationID and the datetime components
df_final_padded = pd.merge(
    master_grid, 
    df_hourly, 
    on=['PULocationID', 'date', 'hour', 'day_of_week', 'month'], 
    how='left'
)

# 2. Fill the missing hours (NaN) with 0 pickups
df_final_padded['pickup_count'] = df_final_padded['pickup_count'].fillna(0).astype(int)

# 3. Final Verification
print(f"Final Dataset Shape: {df_final_padded.shape}")
print(f"Total Unique Zones Represented: {df_final_padded['PULocationID'].nunique()}")
print(f"Percentage of Zero-Pickup Hours: {(df_final_padded['pickup_count'] == 0).mean()*100:.2f}%")



Final Dataset Shape: (568080, 7)
Total Unique Zones Represented: 263
Percentage of Zero-Pickup Hours: 62.30%


In [178]:
# 1. Convert the 'date' column to strings so the Parquet engine can save it
df_final_padded['date'] = df_final_padded['date'].astype(str)

# 2. Now save using fastparquet
df_final_padded.to_parquet(
    '../data/processed/taxi_demand_hourly_padded.parquet', 
    index=False, 
    engine='fastparquet'
)

print("Dataset 1 exported successfully! 'date' column converted to string for compatibility.")

Dataset 1 exported successfully! 'date' column converted to string for compatibility.
